In [8]:
# CELDA 1: Preparación del entorno
import sys
import os

# 1. Clonamos el repositorio
!git clone https://github.com/Santiago-Soria/proyecto-transformacion-texto-imagen.git

# 2. Instalamos las dependencias necesarias
!pip install -q optuna polars transformers torch scikit-learn

# 3. Le decimos a Python dónde buscar los módulos de (src/models)
# Esto es vital para poder importar train_classic y train_transformer
sys.path.append('/content/proyecto-transformacion-texto-imagen')
sys.path.append('/content/proyecto-transformacion-texto-imagen/src/models')

print("✅ Entorno configurado correctamente.")

fatal: destination path 'proyecto-transformacion-texto-imagen' already exists and is not an empty directory.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 12.0 MB/s eta 0:00:00
✅ Entorno configurado correctamente.


In [9]:
# ==========================================
# CELDA 2: Carga de datos
# ==========================================
import polars as pl
import numpy as np
PROJECT_ROOT = '/content/proyecto-transformacion-texto-imagen'

ruta_base = f'{PROJECT_ROOT}/data/processed'

df_train = pl.read_csv(f'{ruta_base}/train.csv')
df_test  = pl.read_csv(f'{ruta_base}/test.csv')

X_train = df_train.get_column('text').to_list()
y_train = df_train.get_column('manual_classification').to_numpy()

X_test  = df_test.get_column('text').to_list()
y_test  = df_test.get_column('manual_classification').to_numpy()

print(f"✓ Train: {len(X_train)} | Test: {len(X_test)}")


✓ Train: 908 | Test: 114


In [10]:
# ==========================================
# CELDA 3: Función objetivo para Optuna
# ==========================================
import torch
import optuna
import numpy as np
import json
import os
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, set_seed
)
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from torch.utils.data import Dataset

set_seed(42)
MODEL_NAME  = "dccuchile/bert-base-spanish-wwm-cased"  # BETO
RESULTS_DIR = f'{PROJECT_ROOT}/reports/optuna'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Dataset reutilizable (igual que en train_transformer.py)
class DepressionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=128,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy':        acc,
        'precision_macro': precision,
        'recall_macro':    recall,
        'f1_macro':        f1,
    }

def objective(trial):
    """Función objetivo de Optuna. Retorna Macro-F1 en el set de validación."""

    # ---- Espacio de búsqueda ----
    lr           = trial.suggest_float('learning_rate', 1e-5, 5e-5, log=True)
    batch_size   = trial.suggest_categorical('batch_size', [8, 16, 32])
    weight_decay = trial.suggest_float('weight_decay', 0.0, 0.1)
    num_epochs   = trial.suggest_int('num_epochs', 2, 6)
    warmup_ratio = trial.suggest_float('warmup_ratio', 0.0, 0.2)

    output_dir = f'{PROJECT_ROOT}/models/checkpoints/optuna_trial_{trial.number}'

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model     = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    train_ds = DepressionDataset(X_train, y_train, tokenizer)
    val_ds   = DepressionDataset(X_test,  y_test,  tokenizer)

    training_args = TrainingArguments(
        output_dir              = output_dir,
        eval_strategy           = 'epoch',
        save_strategy           = 'epoch',
        logging_strategy        = 'epoch',
        learning_rate           = lr,
        per_device_train_batch_size = batch_size,
        per_device_eval_batch_size  = batch_size,
        num_train_epochs        = num_epochs,
        weight_decay            = weight_decay,
        warmup_ratio            = warmup_ratio,
        load_best_model_at_end  = True,
        metric_for_best_model   = 'f1_macro',
        greater_is_better       = True,
        save_total_limit        = 1,
        fp16                    = True,   # ← Activa mixed-precision en T4
        seed                    = 42,
        report_to               = 'none', # Desactiva W&B / TensorBoard
    )

    trainer = Trainer(
        model           = model,
        args            = training_args,
        train_dataset   = train_ds,
        eval_dataset    = val_ds,
        compute_metrics = compute_metrics,
        callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()
    metrics = trainer.evaluate()
    f1 = metrics['eval_f1_macro']

    # Guardar métricas de cada trial
    trial_result = {
        'trial':        trial.number,
        'params':       trial.params,
        'f1_macro':     f1,
        'metrics':      metrics
    }
    with open(f'{RESULTS_DIR}/trial_{trial.number}.json', 'w') as f:
        json.dump(trial_result, f, indent=4)

    print(f"\n✓ Trial {trial.number} | F1-Macro: {f1:.4f} | Params: {trial.params}")
    return f1

print("✓ Función objetivo definida")


✓ Función objetivo definida


In [11]:
# ==========================================
# CELDA 4: Ejecutar Optuna
# ==========================================
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Crear el estudio — maximize porque queremos el mayor F1
study = optuna.create_study(
    direction   = 'maximize',
    study_name  = 'BETO_finetuning_hpo',
    sampler     = optuna.samplers.TPESampler(seed=42)  # Bayesian optimization
)

# Baseline: informar a Optuna de los hiperparámetros originales para partir de ahí
study.enqueue_trial({
    'learning_rate': 2e-5,
    'batch_size':    16,
    'weight_decay':  0.01,
    'num_epochs':    6,
    'warmup_ratio':  0.0
})

# n_trials: Ajusta según el tiempo disponible
# Recomendado en T4: 10-15 trials (~60-90 min)
N_TRIALS = 12

print(f"🔍 Iniciando HPO con Optuna | {N_TRIALS} trials | GPU: {torch.cuda.get_device_name(0)}")
print(f"   Baseline: lr=2e-5, batch=16, wd=0.01, epochs=6 → F1 baseline: 0.7411\n")

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print("\n" + "="*50)
print("✅ OPTIMIZACIÓN COMPLETADA")
print("="*50)
print(f"Mejor Macro-F1: {study.best_value:.4f}")
print(f"Mejores hiperparámetros:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


🔍 Iniciando HPO con Optuna | 12 trials | GPU: Tesla T4
   Baseline: lr=2e-5, batch=16, wd=0.01, epochs=6 → F1 baseline: 0.7411



  0%|          | 0/12 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


✓ Trial 11 | F1-Macro: 0.7897 | Params: {'learning_rate': 2.3048016342698774e-05, 'batch_size': 16, 'weight_decay': 0.06379717552110609, 'num_epochs': 5, 'warmup_ratio': 0.0013564417879888579}

✅ OPTIMIZACIÓN COMPLETADA
Mejor Macro-F1: 0.7897
Mejores hiperparámetros:
  learning_rate: 2.3048016342698774e-05
  batch_size: 16
  weight_decay: 0.06379717552110609
  num_epochs: 5
  warmup_ratio: 0.0013564417879888579
